# DE Challenge 1

> ETL basics with Python and SQL

> Public Dataset: [NYC Taxi Open Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)  



## Libraries

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## Get dataset from source

In [ ]:
DATA_URL    = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
LOCAL_FILE  = "yellow_tripdata_2023-01.parquet"
DB_FILE     = "nyc_taxi.db"
SAMPLE_SIZE = 100_000

print(f'Dataset URL  : {DATA_URL}')
print(f'Local file   : {LOCAL_FILE}')
print(f'Database     : {DB_FILE}')
print(f'Sample size  : {SAMPLE_SIZE:,} rows')

Dataset URL  : https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
Local file   : yellow_tripdata_2023-01.parquet
Database     : nyc_taxi.db
Sample size  : 100,000 rows


---
# 1. EXTRACT
Download the dataset and load it into a DataFrame

In [ ]:
if not os.path.exists(LOCAL_FILE):
    print('Downloading dataset from NYC Taxi (~50MB)...')
    urllib.request.urlretrieve(DATA_URL, LOCAL_FILE)
    print(f'File Downloaded: {LOCAL_FILE}')
else:
    print(f'Local file found: {LOCAL_FILE}')

df_raw = pd.read_parquet(LOCAL_FILE)
print(f'Full dataset shape : {df_raw.shape}')

Local file found: yellow_tripdata_2023-01.parquet
Full dataset shape : (3066766, 19)


In [ ]:
# Preview raw data
print('Raw data preview:')
df_raw.head(3)

Raw data preview:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


In [ ]:
# Data types and null counts
print('Data types & null counts:')
summary = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'nulls': df_raw.isnull().sum(),
    'null_%': (df_raw.isnull().mean() * 100).round(2)
})
summary

Data types & null counts:


,dtype,nulls,null_%
VendorID,int64,0,0.00
tpep_pickup_datetime,datetime64[us],0,0.00
tpep_dropoff_datetime,datetime64[us],0,0.00
passenger_count,float64,71743,2.34
trip_distance,float64,0,0.00
RatecodeID,float64,71743,2.34
store_and_fwd_flag,object,71743,2.34
PULocationID,int64,0,0.00
DOLocationID,int64,0,0.00
payment_type,int64,0,0.00


---
# 2. TRANSFORM
Apply 5 transformations to clean and enrich the data

### A. Rename Columns

In [ ]:
rename_map = {
    'tpep_pickup_datetime'  : 'pickup_datetime',
    'tpep_dropoff_datetime' : 'dropoff_datetime',
    'RatecodeID'            : 'rate_code_id',
    'PULocationID'          : 'pickup_location_id',
    'DOLocationID'          : 'dropoff_location_id',
    'VendorID'              : 'vendor_id',
}

df = df_raw.rename(columns=rename_map)
print(f'Renamed {len(rename_map)} columns')
print(f'New column names: {list(df.columns)}')

Renamed 6 columns
New column names: ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code_id', 'store_and_fwd_flag', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


### B. Type Format

In [ ]:
df['pickup_datetime']  = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])
df['vendor_id']        = df['vendor_id'].astype('Int64')
df['passenger_count']  = df['passenger_count'].astype('Int64')
df['payment_type']     = df['payment_type'].astype('Int64')
df['rate_code_id']     = df['rate_code_id'].astype('Int64')

print('Types applied:')
df[['pickup_datetime','dropoff_datetime','vendor_id','passenger_count','payment_type','rate_code_id']].dtypes

Types applied:


,0
pickup_datetime,datetime64[us]
dropoff_datetime,datetime64[us]
vendor_id,Int64
passenger_count,Int64
payment_type,Int64
rate_code_id,Int64


### C. Data Cleansing

In [ ]:
before = len(df)

df = df[df['trip_distance']  > 0]
df = df[df['fare_amount']    > 0]
df = df[df['passenger_count'] > 0]
df = df[df['pickup_datetime'] < df['dropoff_datetime']]

after = len(df)
print(f'Rows before cleansing : {before:,}')
print(f'Rows removed          : {before - after:,}')
print(f'Rows after cleansing  : {after:,}')

Rows before cleansing : 3,066,766
Rows removed          : 182,601
Rows after cleansing  : 2,884,165


### D. Deduplication

In [ ]:
before = len(df)

df = df.drop_duplicates(
    # Columns that are keys
    subset=['pickup_datetime', 'dropoff_datetime',
            'pickup_location_id', 'dropoff_location_id', 'fare_amount']
).reset_index(drop=True)

after = len(df)
print(f'Duplicates removed : {before - after:,}')
print(f'Rows remaining     : {after:,}')

Duplicates removed : 2
Rows remaining     : 2,884,163


### E. Derived Columns

In [ ]:
df['trip_duration_min'] = (
    (df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds() / 60
).round(2)

df['speed_mph'] = (
    df['trip_distance'] / (df['trip_duration_min'] / 60)
).replace([np.inf, -np.inf], np.nan).round(2)

df['total_per_mile'] = (
    df['total_amount'] / df['trip_distance'].replace(0, np.nan)
).round(2)

df['pickup_hour']        = df['pickup_datetime'].dt.hour
df['pickup_date']        = df['pickup_datetime'].dt.date

print('Derived columns added:')
df[['trip_duration_min','speed_mph','total_per_mile','pickup_hour']].describe().round(2)

Derived columns added:


,trip_duration_min,speed_mph,total_per_mile,pickup_hour
count,2884163.00,2884163.00,2884163.00,2884163.00
mean,15.78,13.34,15.83,14.17
std,43.07,321.47,141.39,5.75
min,0.02,0.00,0.00,0.00
25%,7.20,7.92,8.01,11.00
50%,11.55,10.30,10.92,15.00
75%,18.28,14.20,14.67,19.00
max,6179.40,518811.43,48420.00,23.00


### F. Aggregation

In [ ]:
agg_by_hour = (
    df.groupby('pickup_hour')
    .agg(
        total_trips      = ('vendor_id',        'count'),
        avg_fare         = ('fare_amount',       'mean'),
        avg_distance     = ('trip_distance',     'mean'),
        avg_duration_min = ('trip_duration_min', 'mean'),
        total_revenue    = ('total_amount',      'sum'),
    )
    .round(2)
    .reset_index()
)

print(f'Hourly aggregation table: {len(agg_by_hour)} rows')
agg_by_hour

Hourly aggregation table: 24 rows


,pickup_hour,total_trips,avg_fare,avg_distance,avg_duration_min,total_revenue
0,0,79566,19.80,4.03,15.66,2297224.24
1,1,55565,17.82,3.49,14.89,1461587.50
2,2,38729,16.72,3.22,14.65,961056.89
3,3,25053,17.74,3.51,14.57,651827.71
4,4,15835,22.26,4.68,15.79,498472.72
5,5,16141,26.45,6.42,15.68,591519.05
6,6,39941,22.15,4.78,15.16,1221495.60
7,7,79551,18.92,3.68,15.15,2133844.79
8,8,107898,17.43,3.18,15.70,2710773.86
9,9,122642,17.61,3.09,15.34,3129545.52


In [ ]:
# Final transformed dataset overview
print(f'Final clean and transformed dataset shape: {df.shape}')
df.head(3)

Final clean and transformed dataset shape: (2884163, 24)


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,trip_duration_min,speed_mph,total_per_mile,pickup_hour,pickup_date
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1,0.97,1,N,161,141,2,...,0.0,1.0,14.3,2.5,0.0,8.43,6.90,14.74,0,2023-01-01
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1,1.10,1,N,43,237,1,...,0.0,1.0,16.9,2.5,0.0,6.32,10.44,15.36,0,2023-01-01
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1,2.51,1,N,48,238,1,...,0.0,1.0,34.9,2.5,0.0,12.75,11.81,13.90,0,2023-01-01


---
# 3. LOAD
Create the star schema in SQLite and load all tables.

### Create Table Schema

In [ ]:
SCHEMA_SQL = """
DROP TABLE IF EXISTS fact_trips;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_payment;
DROP TABLE IF EXISTS agg_hourly_trips;

-- Calendar date of each trip
CREATE TABLE dim_date (
    date_id     INTEGER PRIMARY KEY,
    date        TEXT    NOT NULL,
    year        INTEGER NOT NULL,
    month       INTEGER NOT NULL,
    day         INTEGER NOT NULL,
    day_of_week TEXT    NOT NULL,
    is_weekend  INTEGER NOT NULL
);

-- Payment methods
CREATE TABLE dim_payment (
    payment_type_id     INTEGER PRIMARY KEY,
    payment_description TEXT    NOT NULL
);

-- Trips
CREATE TABLE fact_trips (
    trip_id             INTEGER PRIMARY KEY AUTOINCREMENT,
    date_id             INTEGER REFERENCES dim_date(date_id),
    payment_type_id     INTEGER REFERENCES dim_payment(payment_type_id),
    vendor_id           INTEGER,
    pickup_location_id  INTEGER,
    dropoff_location_id INTEGER,
    pickup_hour         INTEGER,
    rate_code_desc      TEXT,
    is_weekend          INTEGER,
    passenger_count     INTEGER,
    trip_distance       REAL,
    fare_amount         REAL,
    tip_amount          REAL,
    total_amount        REAL,
    trip_duration_min   REAL,
    speed_mph           REAL,
    total_per_mile      REAL
);

-- AGG TABLE: Hourly summary
CREATE TABLE agg_hourly_trips (
    pickup_hour         INTEGER PRIMARY KEY,
    total_trips         INTEGER,
    avg_fare            REAL,
    avg_distance        REAL,
    avg_duration_min    REAL,
    total_revenue       REAL
);
"""

conn = sqlite3.connect(DB_FILE)
conn.executescript(SCHEMA_SQL)
conn.commit()
print('Star schema created in', DB_FILE)

Star schema created in nyc_taxi.db


### Load dim_date

In [ ]:
dim_date = df[['pickup_date']].drop_duplicates().copy()
dim_date['date_id']     = range(1, len(dim_date) + 1)
dim_date['date']        = dim_date['pickup_date'].astype(str)
dim_date['year']        = pd.to_datetime(dim_date['pickup_date']).dt.year
dim_date['month']       = pd.to_datetime(dim_date['pickup_date']).dt.month
dim_date['day']         = pd.to_datetime(dim_date['pickup_date']).dt.day
dim_date['day_of_week'] = pd.to_datetime(dim_date['pickup_date']).dt.day_name()
dim_date['is_weekend']  = pd.to_datetime(dim_date['pickup_date']).dt.dayofweek >= 5

dim_date[['date_id','date','year','month','day','day_of_week','is_weekend']].to_sql(
    'dim_date', conn, if_exists='append', index=False
)
print(f'dim_date loaded: {len(dim_date)} rows')
dim_date.head()

dim_date loaded: 35 rows


,pickup_date,date_id,date,year,month,day,day_of_week,is_weekend
0,2023-01-01,1,2023-01-01,2023,1,1,Sunday,True
79,2022-12-31,2,2022-12-31,2022,12,31,Saturday,True
23396,2022-10-25,3,2022-10-25,2022,10,25,Tuesday,False
68375,2023-01-02,4,2023-01-02,2023,1,2,Monday,False
130591,2023-01-03,5,2023-01-03,2023,1,3,Tuesday,False


### Load dim_payment

In [ ]:
# Build dim_payment directly from the mapping dict independent of df column order
payment_map = {
    1: 'Credit Card', 2: 'Cash', 3: 'No Charge',
    4: 'Dispute',     5: 'Unknown', 6: 'Voided Trip'
}
dim_payment = pd.DataFrame([
    {'payment_type_id': k, 'payment_description': v}
    for k, v in payment_map.items()
])

dim_payment.to_sql('dim_payment', conn, if_exists='append', index=False)
print(f'dim_payment loaded: {len(dim_payment)} rows')
dim_payment

dim_payment loaded: 6 rows


,payment_type_id,payment_description
0,1,Credit Card
1,2,Cash
2,3,No Charge
3,4,Dispute
4,5,Unknown
5,6,Voided Trip


### Load fact_trips

In [ ]:
rate_map = {
    1: 'Standard Rate', 2: 'JFK',    3: 'Newark',
    4: 'Nassau/Westchester', 5: 'Negotiated Fare', 6: 'Group Ride'
}

date_map = dict(zip(dim_date['pickup_date'].astype(str), dim_date['date_id']))

fact = df[[
    'vendor_id', 'payment_type', 'pickup_location_id',
    'dropoff_location_id', 'passenger_count', 'trip_distance',
    'fare_amount', 'tip_amount', 'total_amount',
    'trip_duration_min', 'speed_mph', 'total_per_mile',
    'pickup_hour', 'pickup_date'
]].copy()

# Derive remaining columns
fact['date_id']       = df['pickup_date'].astype(str).map(date_map)
fact['rate_code_desc']= df['rate_code_id'].map(rate_map).fillna('Unknown')
fact['is_weekend']    = df['pickup_datetime'].dt.dayofweek >= 5

fact = fact.drop(columns=['pickup_date']).rename(columns={'payment_type': 'payment_type_id'})

fact.to_sql('fact_trips', conn, if_exists='append', index=False)
print(f'fact_trips loaded: {len(fact):,} rows')
fact.head(3)

fact_trips loaded: 2,884,163 rows


,vendor_id,payment_type_id,pickup_location_id,dropoff_location_id,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,trip_duration_min,speed_mph,total_per_mile,pickup_hour,date_id,rate_code_desc,is_weekend
0,2,2,161,141,1,0.97,9.3,0.0,14.3,8.43,6.90,14.74,0,1,Standard Rate,True
1,2,1,43,237,1,1.10,7.9,4.0,16.9,6.32,10.44,15.36,0,1,Standard Rate,True
2,2,1,48,238,1,2.51,14.9,15.0,34.9,12.75,11.81,13.90,0,1,Standard Rate,True


### Load agg_hourly_trips

In [ ]:
agg_by_hour.to_sql('agg_hourly_trips', conn, if_exists='replace', index=False)
conn.close()

print(f'agg_hourly_trips loaded: {len(agg_by_hour)} rows')
print(f'\nDatabase saved: {DB_FILE}')

agg_hourly_trips loaded: 24 rows

Database saved: nyc_taxi.db


---
---


# ANALYTICAL QUERIES

In [ ]:
# Helper function to run queries
def run_query(sql, title=''):
    conn = sqlite3.connect(DB_FILE)
    result = pd.read_sql_query(sql, conn)
    conn.close()
    if title:
        print(f'\n{title}')
        print('─' * 50)
    return result

In [ ]:
# Peak hours by number of trips
q1 = run_query("""
    SELECT
        pickup_hour,
        COUNT(trip_id)                    AS total_trips,
        ROUND(AVG(trip_duration_min), 2)  AS avg_duration_min,
        ROUND(AVG(trip_distance), 2)      AS avg_distance_miles,
        ROUND(SUM(total_amount), 2)       AS total_revenue
    FROM fact_trips
    GROUP BY pickup_hour
    ORDER BY total_trips DESC
    LIMIT 10
""", 'Q1: Top 10 Peak Hours')
q1

In [ ]:
# Top 10 pickup locations
q2 = run_query("""
    SELECT
        pickup_location_id,
        COUNT(trip_id)                AS total_pickups,
        ROUND(AVG(fare_amount), 2)    AS avg_fare,
        ROUND(AVG(trip_distance), 2)  AS avg_distance_miles
    FROM fact_trips
    GROUP BY pickup_location_id
    ORDER BY total_pickups DESC
    LIMIT 10
""", 'Q2: Top 10 Pickup Locations')
q2

In [ ]:
# Weekday vs Weekend comparison
q3 = run_query("""
    SELECT
        CASE WHEN d.is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS period,
        COUNT(f.trip_id)                    AS total_trips,
        ROUND(AVG(f.trip_distance), 2)      AS avg_distance_miles,
        ROUND(AVG(f.trip_duration_min), 2)  AS avg_duration_min,
        ROUND(AVG(f.fare_amount), 2)        AS avg_fare,
        ROUND(AVG(f.speed_mph), 2)          AS avg_speed_mph,
        ROUND(SUM(f.total_amount), 2)       AS total_revenue
    FROM fact_trips f
    JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY d.is_weekend
""", 'Q3: Weekday vs Weekend')
q3